# Executar a CNN propria

Este notebook carrega o melhor modelo corrigido, trata as imagens enviadas e apresenta a probabilidade de endometriose.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd /content/drive/MyDrive/TCC2

In [ ]:
!pip install -q -r "cnn_propria_experimento/requirements.txt"

In [ ]:
!python "cnn_propria_experimento/00_auditar_imagens.py"

In [ ]:
!python "cnn_propria_experimento/01_preparar_dataset.py" --overwrite

In [ ]:
!python "cnn_propria_experimento/02_treinar_cnn_propria.py"

In [ ]:
!python "cnn_propria_experimento/03_avaliar_cnn_propria.py"

In [ ]:
!python "cnn_propria_experimento/04_gerar_figuras_tcc.py"

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from google.colab import files

MODEL_PATH = Path('/content/drive/MyDrive/TCC2/cnn_propria_experimento/resultados/resultados_tcc/best_cnn_propria.keras')
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Modelo nao encontrado: {MODEL_PATH}')

model = tf.keras.models.load_model(MODEL_PATH)
if model.input_shape[1] != model.input_shape[2]:
    raise ValueError(f'Entrada do modelo nao e quadrada: {model.input_shape}')
IMG_SIZE = int(model.input_shape[1])
print(f'Modelo carregado com sucesso. Entrada: {IMG_SIZE} x {IMG_SIZE}.')
print('Saida do modelo: probabilidade de endometriose.')

def crop_black_borders(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image_bgr
    x, y, width, height = cv2.boundingRect(max(contours, key=cv2.contourArea))
    return image_bgr[y:y + height, x:x + width]

def remove_specular_reflections(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY)
    mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
    return cv2.inpaint(image_bgr, mask, 3, cv2.INPAINT_TELEA)

def apply_clahe(image_bgr):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    lightness, channel_a, channel_b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(lightness)
    return cv2.cvtColor(cv2.merge((enhanced, channel_a, channel_b)), cv2.COLOR_LAB2BGR)

def treat_image(image_bgr):
    image_bgr = crop_black_borders(image_bgr)
    image_bgr = remove_specular_reflections(image_bgr)
    return apply_clahe(image_bgr)

In [ ]:
print('Selecione uma ou mais imagens laparoscopicas:')
uploaded = files.upload()

for file_name, file_bytes in uploaded.items():
    encoded = np.frombuffer(file_bytes, dtype=np.uint8)
    original_bgr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    if original_bgr is None:
        print(f'Nao foi possivel ler: {file_name}')
        continue

    treated_bgr = treat_image(original_bgr)
    treated_rgb = cv2.cvtColor(treated_bgr, cv2.COLOR_BGR2RGB)
    resized_rgb = cv2.resize(treated_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    batch = np.expand_dims(resized_rgb.astype(np.float32), axis=0)
    probability = float(model.predict(batch, verbose=0)[0][0])
    predicted_class = 'Com endometriose' if probability >= 0.5 else 'Sem endometriose'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Imagem original')
    axes[0].axis('off')
    axes[1].imshow(treated_rgb)
    axes[1].set_title('Imagem tratada')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Arquivo: {file_name}')
    print(f'Classificacao do modelo: {predicted_class}')
    print(f'Probabilidade estimada de endometriose: {probability:.2%}')
    print('-' * 60)